<a href="https://colab.research.google.com/github/riyanmandanna01/LangGraph-AI-Assistant/blob/main/LANGSMITH_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ================================================================
# LANGGRAPH AI ASSISTANT
# Gemini + Groq Automatic 429 Fallback
# DDGS Web Search + Fact Checking
# LangSmith Tracing
# MANUAL API KEY INPUT
# GOOGLE COLAB - EVERYTHING IN ONE CELL
# ================================================================


# ================================================================
# 1. INSTALL REQUIRED PACKAGES
# ================================================================

!pip install -q -U \
    langgraph \
    langchain \
    langchain-core \
    langchain-google-genai \
    langchain-groq \
    langsmith \
    ddgs


# ================================================================
# 2. IMPORT LIBRARIES
# ================================================================

import os
import re
from typing import TypedDict
from getpass import getpass

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

from langsmith import traceable

from langgraph.graph import StateGraph, START, END

from ddgs import DDGS


# ================================================================
# 3. MANUAL API KEY INPUT
# ================================================================

print("=" * 70)
print("🔐 API KEY CONFIGURATION")
print("=" * 70)
print()
print("Enter your API keys below.")
print("Your keys will NOT be displayed while typing.")
print()


# ------------------------------------------------
# Gemini Main API Key
# ------------------------------------------------

API_KEY_1 = getpass(
    "🔑 Enter Main Gemini API Key: "
).strip()


# ------------------------------------------------
# Gemini Web / Fact Check API Key
# ------------------------------------------------

API_KEY_2 = getpass(
    "🔑 Enter Web/Fact-Check Gemini API Key: "
).strip()


# ------------------------------------------------
# Groq API Key
# ------------------------------------------------

GROQ_API_KEY = getpass(
    "🔑 Enter Groq API Key: "
).strip()


# ------------------------------------------------
# LangSmith API Key
# ------------------------------------------------

LANGSMITH_API_KEY = getpass(
    "🔑 Enter LangSmith API Key: "
).strip()


# ================================================================
# 4. CHECK API KEYS
# ================================================================

missing_keys = []

if not API_KEY_1:
    missing_keys.append("Main Gemini API Key")

if not API_KEY_2:
    missing_keys.append("Web/Fact-Check Gemini API Key")

if not GROQ_API_KEY:
    missing_keys.append("Groq API Key")

if not LANGSMITH_API_KEY:
    missing_keys.append("LangSmith API Key")


if missing_keys:

    raise ValueError(
        "❌ Missing API key(s):\n\n"
        + "\n".join(
            f"• {key}"
            for key in missing_keys
        )
        + "\n\n"
        "Please restart the cell and enter all required keys."
    )


print()
print("✅ All API keys received successfully.")
print()


# ================================================================
# 5. LANGSMITH CONFIGURATION
# ================================================================

LANGSMITH_PROJECT = "LangGraph-AI-Assistant"


# ------------------------------------------------
# Configure LangSmith using manually entered key
# ------------------------------------------------

os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY

# Enable automatic LangChain/LangGraph tracing
os.environ["LANGSMITH_TRACING"] = "true"

# LangSmith project
os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT


print("=" * 70)
print("📊 LANGSMITH")
print("=" * 70)
print("Tracing: ENABLED")
print(f"Project: {LANGSMITH_PROJECT}")
print("=" * 70)


# ================================================================
# 6. CREATE GEMINI MAIN MODEL
# ================================================================

gemini_main = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0.7,
    google_api_key=API_KEY_1
)


# ================================================================
# 7. CREATE GEMINI WEB / FACT-CHECK MODEL
# ================================================================

gemini_web = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0.2,
    google_api_key=API_KEY_2
)


# ================================================================
# 8. CREATE GROQ FALLBACK
# ================================================================

groq_fallback = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2,
    api_key=GROQ_API_KEY
)


# ================================================================
# 9. LANGSMITH CONNECTION TEST
# ================================================================

@traceable(
    name="LangSmith Connection Test",
    run_type="chain"
)
def langsmith_test():

    return "LangSmith tracing is working!"


try:

    test_result = langsmith_test()

    print(
        "✅ LangSmith connection test completed."
    )

except Exception as e:

    print(
        "⚠️ LangSmith test failed:"
    )

    print(e)


# ================================================================
# 10. LANGGRAPH STATE
# ================================================================

class ChatState(TypedDict, total=False):

    question: str

    category: str

    web_results: str

    answer: str

    fact_check: str

    provider: str


# ================================================================
# 11. GEMINI QUOTA ERROR DETECTION
# ================================================================

def is_quota_error(error):

    error_text = str(error).upper()

    quota_keywords = [

        "429",

        "RESOURCE_EXHAUSTED",

        "QUOTA EXCEEDED",

        "RATE LIMIT",

        "TOO MANY REQUESTS",

        "GENERATE_CONTENT_FREE_TIER_REQUESTS"
    ]

    return any(
        keyword in error_text
        for keyword in quota_keywords
    )


# ================================================================
# 12. GEMINI -> GROQ AUTOMATIC FALLBACK
# ================================================================

@traceable(
    name="Gemini to Groq Fallback",
    run_type="chain"
)
def call_with_fallback(
    gemini_model,
    prompt,
    task_name="AI Request"
):

    # ------------------------------------------------------------
    # TRY GEMINI
    # ------------------------------------------------------------

    try:

        response = gemini_model.invoke(
            prompt
        )

        return {
            "content": response.content,
            "provider": "Gemini"
        }


    except Exception as gemini_error:

        # --------------------------------------------------------
        # GEMINI 429
        # --------------------------------------------------------

        if is_quota_error(gemini_error):

            print()
            print(
                f"⚠️ Gemini quota exceeded during {task_name}."
            )

            print(
                "🔄 Automatically switching to Groq..."
            )


            # ----------------------------------------------------
            # TRY GROQ
            # ----------------------------------------------------

            try:

                response = groq_fallback.invoke(
                    prompt
                )

                print(
                    "✅ Groq response generated."
                )

                return {
                    "content": response.content,
                    "provider": "Groq"
                }


            except Exception as groq_error:

                return {
                    "content": (
                        "Both Gemini and Groq failed.\n\n"
                        f"Groq error: {groq_error}"
                    ),
                    "provider": "Error"
                }


        # --------------------------------------------------------
        # OTHER GEMINI ERROR
        # --------------------------------------------------------

        return {
            "content": (
                f"Gemini error during {task_name}:\n"
                f"{gemini_error}"
            ),
            "provider": "Error"
        }


# ================================================================
# 13. CLASSIFY QUESTION
# ================================================================

def classify_question(state: ChatState):

    question = state["question"].strip()

    question_lower = question.lower()


    # ------------------------------------------------------------
    # GREETING
    # ------------------------------------------------------------

    greeting_patterns = [

        r"^\s*hi\s*[!.]*$",

        r"^\s*hello\s*[!.]*$",

        r"^\s*hey\s*[!.]*$",

        r"^\s*good morning\s*[!.]*$",

        r"^\s*good afternoon\s*[!.]*$",

        r"^\s*good evening\s*[!.]*$"
    ]


    if any(
        re.match(
            pattern,
            question_lower
        )
        for pattern in greeting_patterns
    ):

        return {
            "category": "greeting"
        }


    # ------------------------------------------------------------
    # ARITHMETIC
    # ------------------------------------------------------------

    if re.fullmatch(
        r"[0-9+\-*/().%\s]+",
        question
    ):

        return {
            "category": "math"
        }


    # ------------------------------------------------------------
    # GENERAL QUESTION
    # ------------------------------------------------------------

    return {
        "category": "general"
    }


# ================================================================
# 14. ROUTER
# ================================================================

def route_question(state: ChatState):

    category = state["category"]


    if category == "greeting":

        return "greeting"


    if category == "math":

        return "math"


    return "web_search"


# ================================================================
# 15. GREETING NODE
# ================================================================

def greeting_node(state: ChatState):

    answer = (
        "Hello! 😊 Nice to meet you! "
        "I'm your LangGraph AI Assistant. "
        "Ask me anything and I'll do my best to help!"
    )


    return {

        "answer": answer,

        "provider": "Local"
    }


# ================================================================
# 16. MATH NODE
# ================================================================

def math_node(state: ChatState):

    expression = state["question"]


    try:

        result = eval(
            expression,
            {"__builtins__": {}},
            {}
        )


        answer = (
            f"The answer is {result} 😊"
        )


    except Exception:

        answer = (
            "I couldn't calculate that expression. "
            "Please check the expression and try again."
        )


    return {

        "answer": answer,

        "provider": "Local"
    }


# ================================================================
# 17. DDGS WEB SEARCH
# ================================================================

@traceable(
    name="DDGS Web Search",
    run_type="tool"
)
def perform_web_search(question):

    with DDGS() as ddgs:

        results = ddgs.text(
            query=question,
            max_results=5
        )


    formatted_results = []


    for i, result in enumerate(
        results,
        start=1
    ):

        title = result.get(
            "title",
            "No title"
        )

        body = result.get(
            "body",
            "No description"
        )

        url = result.get(
            "href",
            "No URL"
        )


        formatted_results.append(
            f"""
RESULT {i}

Title:
{title}

Description:
{body}

URL:
{url}
"""
        )


    if formatted_results:

        return "\n".join(
            formatted_results
        )


    return "No useful web results were found."


# ================================================================
# 18. WEB SEARCH NODE
# ================================================================

def web_search_node(state: ChatState):

    question = state["question"]


    try:

        web_results = perform_web_search(
            question
        )


    except Exception as e:

        web_results = (
            "Web search failed.\n"
            f"Error: {e}"
        )


    return {

        "web_results": web_results
    }


# ================================================================
# 19. GENERAL AI ANSWER
# ================================================================

def general_answer_node(state: ChatState):

    question = state["question"]

    web_results = state.get(
        "web_results",
        "No web results available."
    )


    prompt = f"""
You are a friendly AI assistant.

Answer the user's question using the
web-search information below.

USER QUESTION:
{question}

WEB SEARCH RESULTS:
{web_results}

Instructions:

1. Give a clear and useful answer.
2. Use simple language.
3. Do not invent facts.
4. If the search results are insufficient,
   clearly say so.
5. Mention important sources when appropriate.
6. Be friendly and conversational.
"""


    result = call_with_fallback(
        gemini_web,
        prompt,
        "Web Answer"
    )


    return {

        "answer": result["content"],

        "provider": result["provider"]
    }


# ================================================================
# 20. FACT CHECK
# ================================================================

def fact_check_node(state: ChatState):

    question = state["question"]

    answer = state.get(
        "answer",
        ""
    )

    web_results = state.get(
        "web_results",
        ""
    )


    prompt = f"""
You are a careful fact-checking assistant.

Check the AI-generated answer against
the provided web-search information.

USER QUESTION:
{question}

AI ANSWER:
{answer}

WEB SEARCH RESULTS:
{web_results}

Return exactly this structure:

VERDICT:
CORRECT / PARTIALLY CORRECT / NEEDS VERIFICATION

CONFIDENCE:
HIGH / MEDIUM / LOW

FACT CHECK:
Explain whether the important claims are supported.

CORRECTIONS:
If anything is incorrect or unsupported,
explain what should be corrected.

Otherwise write:
"No major correction required."
"""


    result = call_with_fallback(
        gemini_web,
        prompt,
        "Fact Check"
    )


    return {

        "fact_check": result["content"]
    }


# ================================================================
# 21. FINAL NODE
# ================================================================

def final_node(state: ChatState):

    answer = state.get(
        "answer",
        "No answer generated."
    )

    fact_check = state.get(
        "fact_check",
        ""
    )

    provider = state.get(
        "provider",
        "Unknown"
    )


    print()
    print("=" * 70)

    print("🤖 AI ANSWER")

    print("=" * 70)

    print(answer)


    print()
    print("-" * 70)

    print(
        f"⚡ AI PROVIDER: {provider}"
    )


    if fact_check:

        print()
        print("=" * 70)

        print("🔎 FACT CHECK")

        print("=" * 70)

        print(fact_check)


    print("=" * 70)


    return state


# ================================================================
# 22. CREATE LANGGRAPH
# ================================================================

graph = StateGraph(ChatState)


# ================================================================
# 23. ADD NODES
# ================================================================

graph.add_node(
    "classify",
    classify_question
)

graph.add_node(
    "greeting",
    greeting_node
)

graph.add_node(
    "math",
    math_node
)

graph.add_node(
    "web_search",
    web_search_node
)

graph.add_node(
    "general",
    general_answer_node
)

graph.add_node(
    "fact_check",
    fact_check_node
)

graph.add_node(
    "final",
    final_node
)


# ================================================================
# 24. GRAPH EDGES
# ================================================================

graph.add_edge(
    START,
    "classify"
)


graph.add_conditional_edges(
    "classify",
    route_question,
    {
        "greeting": "greeting",
        "math": "math",
        "web_search": "web_search"
    }
)


graph.add_edge(
    "greeting",
    "final"
)


graph.add_edge(
    "math",
    "final"
)


graph.add_edge(
    "web_search",
    "general"
)


graph.add_edge(
    "general",
    "fact_check"
)


graph.add_edge(
    "fact_check",
    "final"
)


graph.add_edge(
    "final",
    END
)


# ================================================================
# 25. COMPILE LANGGRAPH
# ================================================================

app = graph.compile()


# ================================================================
# 26. DISPLAY SYSTEM
# ================================================================

print()
print("=" * 70)

print("🧠 LANGGRAPH AI ASSISTANT")

print("=" * 70)

print("""
FLOW:

START
  |
  ▼
CLASSIFY
  |
  ├── GREETING ──────────────► FINAL
  |
  ├── MATH ──────────────────► FINAL
  |
  └── WEB SEARCH
          |
          ▼
       DDGS SEARCH
          |
          ▼
        GEMINI
          |
       429 ERROR?
        /      \\
      NO        YES
       |         |
       ▼         ▼
    GEMINI      GROQ
       |         |
       └────┬────┘
            ▼
          ANSWER
            |
            ▼
        FACT CHECK
            |
            ▼
          FINAL
            |
            ▼
           END


LANGSMITH TRACE:

LangGraph AI Assistant
        |
        ├── Classify
        |
        ├── DDGS Web Search
        |
        ├── Generate Answer
        │       |
        │       ├── Gemini
        │       └── Groq fallback
        |
        ├── Fact Check
        │       |
        │       ├── Gemini
        │       └── Groq fallback
        |
        └── Final
""")


print("=" * 70)

print("🔑 Main Gemini API      = Gemini Answer")
print("🔑 Web Gemini API       = Web Answer + Fact Check")
print("🔑 Groq API             = Gemini 429 Fallback")
print("🔑 LangSmith API        = Tracing")

print()
print(f"📊 LangSmith Project: {LANGSMITH_PROJECT}")
print("📡 LangSmith Tracing: ENABLED")

print("=" * 70)


# ================================================================
# 27. CONTINUOUS CHAT
# ================================================================

while True:

    print()

    question = input(
        "You: "
    ).strip()


    # ------------------------------------------------------------
    # EXIT
    # ------------------------------------------------------------

    if question.lower() in [
        "exit",
        "quit",
        "bye"
    ]:

        print(
            "\nAI: Goodbye! 👋 Have a great day!"
        )

        break


    # ------------------------------------------------------------
    # EMPTY INPUT
    # ------------------------------------------------------------

    if not question:

        print(
            "AI: Please enter a question 😊"
        )

        continue


    # ------------------------------------------------------------
    # LANGGRAPH EXECUTION
    # ------------------------------------------------------------

    try:

        result = app.invoke(

            {
                "question": question
            },

            config={

                "run_name":
                    "LangGraph AI Assistant",

                "tags": [
                    "langgraph",
                    "gemini",
                    "groq",
                    "web-search",
                    "fact-check"
                ],

                "metadata": {

                    "application":
                        "LangGraph AI Assistant",

                    "environment":
                        "Google Colab"
                }
            }
        )


    except Exception as e:

        print()
        print("❌ Unexpected error:")
        print(e)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 812.1/812.1 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 9.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



⚠️ Gemini quota exceeded during Web Answer.
🔄 Automatically switching to Groq...
✅ Groq response generated.


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



⚠️ Gemini quota exceeded during Fact Check.
🔄 Automatically switching to Groq...
✅ Groq response generated.

🤖 AI ANSWER
The current Prime Minister of India is **Narendra Modi**. He has been in office since 26 May 2014 and was re‑elected for a third term after the 2024 general elections【Result 1】.  

*Source: Wikipedia entry on Narendra Modi and the “Prime Minister of India” page, which both note that he is the incumbent prime minister.*

----------------------------------------------------------------------
⚡ AI PROVIDER: Groq

🔎 FACT CHECK
VERDICT:
CORRECT

CONFIDENCE:
HIGH

FACT CHECK:
- The AI states that Narendra Modi is the current Prime Minister of India. All web‑search results (Result 1, Result 2, Result 3, Result 4, Result 5) confirm that Narendra Modi has been serving as Prime Minister since 26 May 2014.
- The AI adds that he was re‑elected for a third term after the 2024 general elections. Results 1, 3, 4 and 5 note that he was sworn in for his third term on 9 June 2024 fol

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



⚠️ Gemini quota exceeded during Web Answer.
🔄 Automatically switching to Groq...
✅ Groq response generated.


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



⚠️ Gemini quota exceeded during Fact Check.
🔄 Automatically switching to Groq...
✅ Groq response generated.

🤖 AI ANSWER
The current President of the United States is **Donald John Trump**【4†L1-L2】【3†L1-L2】. He is serving as the 47th president.

----------------------------------------------------------------------
⚡ AI PROVIDER: Groq

🔎 FACT CHECK
VERDICT:
CORRECT

CONFIDENCE:
HIGH

FACT CHECK:
The AI answer states that the current President of the United States is Donald John Trump and that he is the 47th president. The provided web‑search results support this claim:
- Result 2 (List of presidents of the United States) notes that “the incumbent president is Donald Trump.”
- Result 3 (President Donald J. Trump) describes Trump as the president after a 2024 election victory.
- Result 4 (USAGov) explicitly says “The 47th and current president of the United States is Donald John Trump.”
These sources collectively confirm both parts of the AI’s statement.

CORRECTIONS:
No major correctio